# Post-Harvest Intelligence: train the storage-risk model on real datasets

**Goal:** predict *how many days remain before spoilage risk becomes high* from crop, harvest date, and temperature/humidity readings.

```
 Real sensor data (Uganda, India) ─► realistic temperature/humidity patterns ─┐
 Real outcomes (avocado ripening)  ─► calibrates how fast heat ages a crop    ├─► training set ─► model ─► JSON for the HTML app
 Crop tables (USDA / ASHRAE / IIR) ─► safe limits per crop (labels' rules)    ┘
```

**How to use:** download the datasets (links below), put them in the folders shown, then run all cells. Any missing dataset is skipped with a message, so you can start with one.

## 1. Datasets to download

| Dataset | Link | Put it in | What we use it for |
|---|---|---|---|
| Uganda fruit cold storage (temp, humidity, light, CO₂, Good/Bad) | https://data.mendeley.com/datasets/czz68d9fwj/1 | `data/uganda/` | real sensor patterns + a real-label sanity check |
| India grain storage (hourly temp, humidity, moisture, CO₂, mold/insect risk; **synthetic**, literature-based) | https://aikosh.indiaai.gov.in/home/datasets/details/grain_storage_deterioration_risk_dataset_mold_and_insect_infestation.html | `data/india/` (the `dataset.json`) | grain temperature/humidity patterns |
| Hass avocado ripening (spreadsheet + photos) | https://data.mendeley.com/datasets/3xd9n945v8/1 | `data/avocado/` (only the Excel/CSV file is needed, skip the 14,710 photos) | real aging speed at 10 °C vs 20 °C |
| Apple cold room time series (optional) | https://data.mendeley.com/datasets/h4sghvyjyt/2 | `data/apple/` | not used yet, useful for condensation work |

**Knowledge tables** (the crop limits in section 4 come from these, check the values there against them):
- IIR/ASHRAE table, 200+ cultivars: https://iifiir.org/en/encyclopedia-of-refrigeration/cold-storage-conditions-for-fruits-and-vegetables
- USDA Agriculture Handbook 66: https://www.ars.usda.gov/is/np/CommercialStorage/CommercialStorage.pdf
- UGA shelf life and humidity table: https://openscholar.uga.edu/record/21833/files/FS100.pdf
- UMaine storage conditions: https://extension.umaine.edu/publications/4135e/
- FAO postharvest manual (chilling injury): https://www.fao.org/3/ae075e/ae075e14.htm

Mendeley and AIKosh need a manual download (zip, free). Check each dataset's licence and cite it in your report.

In [ ]:
%pip install -q scikit-learn pandas numpy joblib openpyxl matplotlib

In [ ]:
import json, re, glob, warnings
from pathlib import Path
import numpy as np, pandas as pd, joblib, matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, f1_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier, export_text
warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)
DATA = Path("data"); OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
def pick(cols, *keys):
    for k in keys:
        for c in cols:
            if k in str(c).lower(): return c

## 2. Uganda dataset: load, clean, explore

In [ ]:
def load_uganda():
    files = list(DATA.glob("uganda/**/*.csv"))
    if not files: print("Uganda CSV not found in data/uganda/ -> skipped"); return None
    d = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    d.columns = [c.strip().lower() for c in d.columns]
    d = d.rename(columns={"humidity": "humid", "temperature": "temp"})
    for c in ["temp", "humid", "light", "co2"]: d[c] = pd.to_numeric(d[c], errors="coerce")
    d["fruit"] = d["fruit"].astype(str).str.strip().str.lower()
    d["bad"] = (d["class"].astype(str).str.strip().str.lower() == "bad").astype(int)
    n0 = len(d)
    d = d.dropna(subset=["temp", "humid", "co2", "light"])
    d = d[d.temp.between(-10, 60) & d.humid.between(0, 100) & d.co2.between(0, 10000)]
    print(f"Uganda: {n0} rows, {len(d)} kept after cleaning")
    return d.reset_index(drop=True)
ug = load_uganda()
if ug is not None:
    display(ug.groupby("fruit")[["temp", "humid", "co2", "bad"]].agg(["mean", "min", "max"]).round(1))
    print("Share of 'Bad':", ug.bad.mean().round(3))

### 2b. Can a model tell Good from Bad conditions?
The label describes *conditions*, not a rotten batch. If it is a simple threshold rule, the tree below will show it, and a high score is then expected and not impressive. Split is by time order inside each fruit, so training rows never come after test rows.

In [ ]:
if ug is not None:
    ug["rank"] = ug.groupby("fruit").cumcount() / ug.groupby("fruit")["fruit"].transform("count")
    X = pd.get_dummies(ug[["temp", "humid", "co2", "light", "fruit"]], columns=["fruit"]).astype(float)
    tr, te = ug["rank"] < .7, ug["rank"] >= .7
    clf = HistGradientBoostingClassifier(random_state=0).fit(X[tr], ug.bad[tr])
    p = clf.predict(X[te])
    print(f"accuracy {accuracy_score(ug.bad[te], p):.3f} | F1 {f1_score(ug.bad[te], p):.3f} | majority baseline {max(ug.bad[te].mean(), 1-ug.bad[te].mean()):.3f}")
    tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(ug[["temp", "humid", "co2", "light"]], ug.bad)
    print(export_text(tree, feature_names=["temp", "humid", "co2", "light"]))

## 3. India grain dataset: load and find the columns
The file structure was not inspectable when this notebook was written, so columns are found by keyword. **Check the printout and override `IN` if a column is wrong.**

In [ ]:
def load_india():
    f = list(DATA.glob("india/**/*.json"))
    if not f: print("India dataset.json not found in data/india/ -> skipped"); return None, {}
    raw = json.load(open(f[0]))
    if isinstance(raw, dict):
        lists = [v for v in raw.values() if isinstance(v, list)]
        raw = lists[0] if lists else [raw]
    d = pd.json_normalize(raw)
    cols = list(d.columns); print("Columns:", cols)
    IN = {"T": pick(cols, "grain_temp", "temp"), "RH": pick(cols, "humid", "rh"),
          "crop": pick(cols, "grain", "crop", "commodity"), "risk": pick(cols, "mold_risk", "risk")}
    return d, IN
india, IN = load_india()
# IN = {"T": "your_temp_col", "RH": "your_humidity_col", "crop": "your_grain_col", "risk": "your_risk_col"}   # override here
print(IN)
if india is not None: display(india.head())

## 4. Crop knowledge and calibration
Each crop: best temperature, humidity limit (minimum for fresh produce = water loss, maximum for dry crops = mold), days of life at the best conditions, chilling limit, and **Q10** (how many times faster it ages per +10 °C). Values are approximate: verify against the tables in section 1.

In [ ]:
# name: (Topt C, RH limit %, life at optimum days, chilling limit C, Q10, dry crop?)
CROPS = {"tomato":(12,90,14,10,2.5,0),"potato":(7,92,150,4,2.0,0),"banana":(14,90,14,12,3.0,0),
 "apple":(1,90,150,-1,2.2,0),"orange":(5,90,60,3,2.2,0),"strawberry":(0,90,7,-0.5,3.5,0),
 "lettuce":(0,95,14,-0.5,3.0,0),"carrot":(0,95,150,-1,2.0,0),"cabbage":(0,95,120,-1,2.0,0),
 "pepper":(8,90,14,7,2.8,0),"olive":(7,90,40,5,2.5,0),"pineapple":(10,90,28,8,2.5,0),
 "onion":(0,70,180,-1,2.0,1),"garlic":(0,70,200,-1,2.0,1),"wheat":(15,65,365,-5,2.5,1),
 "maize":(15,70,300,-5,2.5,1),"rice":(15,70,365,-5,2.5,1),"dates":(5,65,300,-5,2.0,1),"pumpkin":(12,70,100,10,2.2,1)}

def calibrate_avocado():
    f = list(DATA.glob("avocado/**/*.xlsx")) + list(DATA.glob("avocado/**/*.csv"))
    if not f: print("Avocado spreadsheet not found -> default avocado values"); return None
    d = pd.read_excel(f[0]) if f[0].suffix == ".xlsx" else pd.read_csv(f[0])
    cols = list(d.columns); print("Avocado columns:", cols)
    g, s, st, dy = pick(cols, "storage", "group"), pick(cols, "sample"), pick(cols, "index", "stage", "ripen"), pick(cols, "day")
    d[st] = pd.to_numeric(d[st], errors="coerce"); d[dy] = pd.to_numeric(d[dy], errors="coerce")
    d["temp"] = d[g].astype(str).str.extract(r"(10|20)")[0].astype(float)
    end = d[d[st] >= 4].groupby([s, "temp"])[dy].min().reset_index()      # first day at end of shelf life
    med = end.groupby("temp")[dy].median(); print("Median days to end of shelf life:", med.to_dict())
    if 10 in med and 20 in med and med[20] > 0:
        return {"t10": med[10], "Q10": float(np.clip(med[10] / med[20], 1.2, 5))}
av = calibrate_avocado()
t10, q10 = (av["t10"], av["Q10"]) if av else (14, 2.5)   # fallback guesses if the data is missing
CROPS["avocado"] = (6, 85, round(t10 * q10 ** 0.4, 1), 5, round(q10, 2), 0)   # life at 6 C from the 10 C measurement
print("avocado:", CROPS["avocado"])

## 5. Build the training set
Real temperature/humidity readings (Uganda, India) are reused as realistic scenarios for the matching crops. The *label* (days left) comes from the crop rules and the calibrated aging speed, with noise for unknown past conditions.

In [ ]:
pools = {}
if ug is not None:
    for c in CROPS:
        s = ug[ug.fruit.str.contains(rf"\b{c}\b")]
        if len(s): pools[c] = s[["temp", "humid"]].to_numpy()
if india is not None and IN.get("T") and IN.get("RH") and IN.get("crop"):
    names = india[IN["crop"]].astype(str).str.lower()
    for c in ["wheat", "rice", "maize"]:
        s = india[names.str.contains(c if c != "rice" else "rice|paddy")]
        t, r = pd.to_numeric(s[IN["T"]], errors="coerce"), pd.to_numeric(s[IN["RH"]], errors="coerce")
        ok = t.notna() & r.notna()
        if ok.any(): pools[c] = np.c_[t[ok], r[ok]]
print("Crops with real sensor patterns:", {k: len(v) for k, v in pools.items()})

def life(P, T, RH):
    Topt, lim, L0, Tmin, Q10, dry = P.T
    L = L0 * Q10 ** (-np.maximum(T - Topt, 0) / 10) * np.where(T < Tmin, np.exp(-.2 * (Tmin - T)), 1)
    L = L * np.where(dry == 1, np.where(RH > lim, np.exp(-(RH - lim) / 6), 1), np.where(RH < lim, np.exp(-(lim - RH) / 12), 1))
    return np.maximum(L, .5)

COLS = ["T","RH","Th","RHh","age","Topt","RHlim","L0","Tmin","Q10","dry","dT","dRH","dTh","dRHh","agef"]
def make(n, names):
    nm = rng.choice(names, n); P = np.array([CROPS[x] for x in nm], float); Topt, lim, L0, Tmin, Q10, dry = P.T
    T = rng.uniform(Topt - 6, Topt + 25); RH = rng.uniform(35, 100, n)
    for c in names:
        if c in pools:
            i = np.where((nm == c) & (rng.random(n) < .5))[0]; s = pools[c][rng.integers(len(pools[c]), size=len(i))]
            T[i] = s[:, 0] + rng.normal(0, .5, len(i)); RH[i] = np.clip(s[:, 1] + rng.normal(0, 1, len(i)), 25, 100)
    Tp = T + rng.normal(0, 2.5, n); RHp = np.clip(RH + rng.normal(0, 6, n), 25, 100)
    Lp, Ln = life(P, Tp, RHp), life(P, T, RH)
    age = np.minimum(rng.uniform(0, 1.2, n) * Lp, 400); left = np.maximum(0, 1 - age / Lp) * Ln
    Th = Tp + rng.normal(0, .5, n); RHh = np.clip(RHp + rng.normal(0, 1.5, n), 25, 100)   # logged averages since harvest
    sg = lambda r: np.where(dry == 1, r - lim, lim - r)
    X = pd.DataFrame(dict(T=T, RH=RH, Th=Th, RHh=RHh, age=age, Topt=Topt, RHlim=lim, L0=L0, Tmin=Tmin, Q10=Q10, dry=dry,
                          dT=T - Topt, dRH=sg(RH), dTh=Th - Topt, dRHh=sg(RHh), agef=age / L0))
    return nm, X[COLS], left
names = list(CROPS); nm, X, y = make(80000, names)
print(X.shape); display(X.describe().T[["mean", "min", "max"]].round(1))

## 6. Train, compare, keep the best
Two checks: crops seen in training, and **crops held out completely** (harder, closer to adding a new crop later).

In [ ]:
risk = lambda d: np.where(d <= 3, 2, np.where(d <= 10, 1, 0))      # high <=3 days, medium <=10 days
hold = [c for c in ["carrot", "banana", "maize"] if c in names]
test = np.isin(nm, hold); rest = np.where(~test)[0]; rng.shuffle(rest); val = np.zeros(len(nm), bool); val[rest[:int(.15 * len(rest))]] = True
trn = ~test & ~val
def report(m, mask, tag):
    p = np.expm1(m.predict(X[mask])).clip(0); t = y[mask]; hi = t <= 3
    r = dict(tag=tag, R2_log=r2_score(np.log1p(t), np.log1p(p)), MAE_days=mean_absolute_error(t, p),
             risk_acc=(risk(p) == risk(t)).mean(), high_caught=(risk(p[hi]) >= 1).mean())
    print({k: (round(v, 3) if not isinstance(v, str) else v) for k, v in r.items()}); return r
cands = {"GBM d5": GradientBoostingRegressor(n_estimators=150, max_depth=5, learning_rate=.12, subsample=.8, random_state=0),
         "GBM d6": GradientBoostingRegressor(n_estimators=200, max_depth=6, learning_rate=.1, subsample=.8, random_state=0),
         "RandomForest": RandomForestRegressor(n_estimators=150, min_samples_leaf=5, n_jobs=-1, random_state=0)}
res = {}
for k, m in cands.items():
    m.fit(X[trn], np.log1p(y[trn])); print("\n", k); report(m, val, "validation"); res[k] = report(m, test, "held-out crops")
best = max(res, key=lambda k: res[k]["R2_log"]); print("\nBest model:", best)

**Reading the scores:** the labels come from the simulator with noise, so a score near 0.9+ means the model learned the crop rules well. It does **not** prove accuracy on real batches. Section 7 is the only check against real labels. If the best score is under about 0.9, raise `n_estimators`, lower the noise in `make`, or add crops.

## 7. Sanity check against real labels (Uganda Good/Bad)
For Uganda readings, ask the model how many days remain (assume a 3-day-old batch kept at those conditions). Shorter time should go with 'Bad'. AUC 0.5 = no signal, 1.0 = perfect.

In [ ]:
model = cands[best]
if ug is not None:
    rows = []
    for c in names:
        s = ug[ug.fruit.str.contains(rf"\b{c}\b")]
        if len(s) == 0: continue
        P = np.tile(CROPS[c], (len(s), 1)).astype(float); Topt, lim, L0, Tmin, Q10, dry = P.T
        T, RH, age = s.temp.to_numpy(), s.humid.to_numpy(), np.full(len(s), 3.0)
        sg = lambda r: np.where(dry == 1, r - lim, lim - r)
        F = pd.DataFrame(dict(T=T, RH=RH, Th=T, RHh=RH, age=age, Topt=Topt, RHlim=lim, L0=L0, Tmin=Tmin, Q10=Q10, dry=dry,
                              dT=T - Topt, dRH=sg(RH), dTh=T - Topt, dRHh=sg(RH), agef=age / L0))[COLS]
        d = np.expm1(model.predict(F)).clip(0)
        print(f"{c:12s} rows {len(s):5d}  AUC {roc_auc_score(s.bad, -d):.3f}" if s.bad.nunique() > 1 else f"{c}: only one class")
        rows.append(pd.DataFrame({"days": d, "bad": s.bad.to_numpy()}))
    if rows:
        a = pd.concat(rows); a.groupby("bad").days.hist(alpha=.5, bins=30); plt.legend(["Good", "Bad"]); plt.xlabel("predicted days left"); plt.show()

## 8. Save the model and rebuild the HTML app
Final fit on all crops, then export `joblib` (Python) and `JSON` (web). Put `post_harvest_preview.html` in this folder to regenerate it with the new model. The JSON export works for the gradient boosting models; if RandomForest won, re-run with `best = 'GBM d5'`.

In [ ]:
assert best.startswith("GBM"), "JSON/HTML export needs a GBM model: set best = 'GBM d6' (or 'GBM d5') and re-run this cell"
final = cands[best].fit(X, np.log1p(y))
joblib.dump({"model": final, "features": COLS, "crops": CROPS}, OUT / "post_harvest_model.joblib")
trees = [{"f": e.tree_.feature.tolist(), "t": [float(f"{x:.6g}") for x in e.tree_.threshold], "l": e.tree_.children_left.tolist(),
          "r": e.tree_.children_right.tolist(), "v": [round(float(x[0][0]), 4) for x in e.tree_.value]} for e in final.estimators_[:, 0]]
M = {"init": float(final.init_.constant_[0][0]), "lr": final.learning_rate, "trees": trees,
     "crops": {k: {"Topt": v[0], "rhlim": v[1], "L0": v[2], "Tmin": v[3], "Q10": v[4], "dry": v[5]} for k, v in CROPS.items()}}
js = json.dumps(M, separators=(",", ":")); (OUT / "post_harvest_model.json").write_text(js)
# check the export reproduces sklearn
x = X.iloc[:200].to_numpy()
def run(r):
    s = M["init"]
    for t in trees:
        n = 0
        while t["l"][n] >= 0: n = t["l"][n] if r[t["f"][n]] <= t["t"][n] else t["r"][n]
        s += M["lr"] * t["v"][n]
    return s
print("export max diff:", np.abs(final.predict(X.iloc[:200]) - np.array([run(r) for r in x])).max())
h = Path("post_harvest_preview.html")
if h.exists():
    html = re.sub(r"const M=.*?;const \$=", lambda m: "const M=" + js + ";const $=", h.read_text(), count=1, flags=re.S)
    (OUT / "post_harvest_preview.html").write_text(html); print("HTML rebuilt in outputs/")
else: print("Copy post_harvest_preview.html next to this notebook to rebuild the app")
print(f"Saved in {OUT.resolve()}")

## 9. Next steps
1. Replace the approximate crop values in section 4 with the exact numbers from the IIR/USDA tables for your crops.
2. Log your own sensor readings and ask farmers *"did the batch stay good?"*. Add that as real labels and retrain: this is what turns a rule-based model into a learned one.
3. Add real spoilage outcomes for more crops as you find them. Each added crop with real outcomes improves the calibration in section 4.